In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
# read the mca data with SOC codes and the SOC-AIOE mapping
mca_df = pd.read_csv('../data/auxiliary/final_mca_soc_code.csv')
mca_df['SOC Codes'] = mca_df['SOC Codes'].apply(ast.literal_eval)
# aioe map
soc_aioe_df = pd.read_csv('../data/ai_measurements/soc_aioe.csv')
soc_aioe_map = dict(zip(soc_aioe_df.SOC, soc_aioe_df.AIOE))

# comple map
soc_comple_df = pd.read_csv('../data/ai_measurements/soc_comple.csv')
soc_comple_map = dict(zip(soc_comple_df.SOC, soc_comple_df.Complementarity))

In [3]:
def apply_mapping(codes, mapping):
    """
        Given a list of codes and their mapping to AIOE or comple, 
        return the mapping of the first available score.
        
        If there is no best option, then we have to just look into 
        similar jobs (those sharing the first four digits) and average their scores
    """
    for code in codes:
        score = mapping.get(code, np.nan)

        if not np.isnan(score):
            return score

    # Fallback where we average jobs' score sharing first 4 digits
    fallback_scores = []

    code = codes[0]
    prefix = str(code)[:4]

    for mapped_code, score in mapping.items():
        if mapped_code.startswith(prefix) and not np.isnan(score):
            fallback_scores.append(score)

    if fallback_scores:
        return np.mean(fallback_scores)

    return np.nan

In [4]:
mca_df['AIOE'] = mca_df['SOC Codes'].apply(
    lambda codes : apply_mapping(codes, soc_aioe_map)
)
mca_df['Complementarity'] = mca_df['SOC Codes'].apply(
    lambda codes : apply_mapping(codes, soc_comple_map)
)

In [5]:
mca_df.to_csv('../data/auxiliary/mca_ai_measurements.csv', index=False)